In [1]:
!pip install -q -U langgraph langchain-core openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 32.4 MB/s eta 0:00:00


In [2]:
# Import os module
import os

# Used for type definition of our graph state
from typing import TypedDict

# Import LangGraph components
from langgraph.graph import StateGraph, START, END

# Import OpenAI client
# OpenRouter provides an OpenAI-compatible API
from openai import OpenAI

# Import Google Colab userdata
# This allows us to safely read the API key from Colab Secrets
from google.colab import userdata

In [3]:
# Get OpenRouter API key from Google Colab Secrets
OPENROUTER_API_KEY = userdata.get("GenAiChatbot")

# Check whether API key was found
if not OPENROUTER_API_KEY:
    raise ValueError(
        "OPENROUTER_API_KEY not found. "
        "Please add it to Google Colab Secrets."
    )

print("OpenRouter API key loaded successfully.")

OpenRouter API key loaded successfully.


In [4]:
# Create OpenRouter client
# OpenRouter provides an OpenAI-compatible API

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

print("OpenRouter client created successfully.")

OpenRouter client created successfully.


In [5]:
# Select the model that will generate the response

MODEL_NAME = "openai/gpt-oss-20b"

print("Model selected:", MODEL_NAME)

Model selected: openai/gpt-oss-20b


In [6]:
# ============================================================
# 1. DEFINE THE GRAPH STATE SCHEMA
# ============================================================

class AgentState(TypedDict):

    # Stores the question given by the user
    user_query: str

    # Stores the response generated by the LLM
    response: str

    # Stores whether the response passed validation
    is_valid: bool

In [7]:
# ============================================================
# 2. DEFINE NODE FUNCTIONS
# ============================================================

# Generator Node
# This node sends the user's query to OpenRouter
# and generates an answer using the LLM.

def generate_response_node(state: AgentState):

    # Get user's question from the graph state
    query = state["user_query"]

    # Create a prompt for the LLM
    prompt = f"""
You are a helpful AI assistant.

Answer the following user question clearly and
in a simple way.

User Question:
{query}

Give a useful and meaningful answer.
"""

    # Send the prompt to OpenRouter
    response = client.chat.completions.create(

        # Select the OpenRouter model
        model=MODEL_NAME,

        # Send messages to the LLM
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],

        # Lower temperature gives more consistent answers
        temperature=0.3
    )

    # Extract the generated text
    generated_answer = response.choices[0].message.content

    # Return updated state
    return {
        "response": generated_answer
    }

In [8]:
# ============================================================
# VALIDATION NODE
# ============================================================

def validation_node(state: AgentState):

    # Get the generated response
    response = state["response"]

    # Check whether response contains more than 10 characters
    valid = len(response) > 10

    # Return validation result
    return {
        "is_valid": valid
    }

In [9]:
# ============================================================
# 3. DEFINE ROUTING LOGIC
# ============================================================

def router(state: AgentState):

    # Check validation result
    if state["is_valid"]:

        # Response is valid
        return "approved"

    else:

        # Response is not valid
        return "rejected"

In [10]:
# ============================================================
# 4. BUILD THE LANGGRAPH STATE MACHINE
# ============================================================

# Create StateGraph using our AgentState
builder = StateGraph(AgentState)

In [11]:
# Add Generator Node
builder.add_node(
    "generator",
    generate_response_node
)

# Add Validator Node
builder.add_node(
    "validator",
    validation_node
)

In [12]:
# Start the graph with the generator node

builder.add_edge(
    START,
    "generator"
)

In [13]:
# After generating the response,
# send it to the validation node

builder.add_edge(
    "generator",
    "validator"
)

In [14]:
# ============================================================
# CONDITIONAL ROUTING
# ============================================================

builder.add_conditional_edges(

    # Routing starts from validator node
    "validator",

    # Function that decides where to go
    router,

    # Possible routes
    {
        # If approved → finish the graph
        "approved": END,

        # If rejected → go back to generator
        "rejected": "generator"
    }
)

In [15]:
# ============================================================
# COMPILE LANGGRAPH APPLICATION
# ============================================================

app = builder.compile()

print("LangGraph compiled successfully!")

LangGraph compiled successfully!


In [16]:
# ============================================================
# 5. EXECUTE GRAPH
# ============================================================

# Ask the user for a question

user_question = input(
    "Enter your question: "
)


# Execute LangGraph

output = app.invoke({

    "user_query": user_question,

    "response": "",

    "is_valid": False
})


# Display result

print(
    "\n--- LANGGRAPH EXECUTION COMPLETE ---"
)

print(
    "\nFinal State:"
)

print(output)


print(
    "\nAI Response:"
)

print(
    output["response"]
)

Enter your question: who is harshad mehta?

--- LANGGRAPH EXECUTION COMPLETE ---

Final State:
{'user_query': 'who is harshad mehta?', 'response': '**Harshad Mehta** was an Indian stockbroker who became famous (and infamous) for a huge securities scam in the early 1990s.\n\n- **Early life** – Born in 1954 in Gujarat, India. He started trading in the Bombay Stock Exchange (now BSE) in the 1980s.\n- **Rise to fame** – In the early 1990s he was called the “Stock Market Wizard” because he seemed to move markets with his trades.\n- **The scam** – In 1992 he used illegal accounting tricks and forged bank receipts to get money from banks and then used that money to buy shares. The scheme created a huge bubble in the market, which burst and caused a crash.\n- **Aftermath** – He was arrested, tried, and convicted for fraud. He spent several years in prison.\n- **Death** – Harshad Mehta died in 2001 at the age of 47.\n- **Legacy** – His case led to major reforms in India’s financial and regulato